# TensorBoard-Plots der rekonstruierten MoE-Budget-Loss-Expertanalysis

Dieses Notebook liest alle TensorBoard-Scalar-Logs aus `moe_runs/randomsearch_new_budgetloss_expertanalysis_reconstructed` und erzeugt PDF-Abbildungen fuer den Anhang der Masterarbeit:

- Genauigkeitsverläufe für Hard- und Top-k-Routing in einer gemeinsamen Abbildung,
- Nutzung von `$f_{small}$`, `$f_{mid}$` und `$f_{large}$` bei `val_hard`.

Die Legenden enthalten den aus TensorBoard gelesenen Budget-Loss-Weight `w_budget`. Die Abbildungen werden unter `moe/plots/figures_budgetloss_expertanalysis_reconstructed` gespeichert.

In [ ]:
from pathlib import Path
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, MultipleLocator, PercentFormatter
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# Einheitlicher Plot-Stil der Masterarbeit.
plt.rcParams.update({
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "lines.linewidth": 1.0,
    "lines.markersize": 3.5,
})

# Beide erzeugten Plots werden über die volle Textbreite eingebunden.
# Wegen der Run-Legende unterhalb ist etwas zusätzliche Höhe vorgesehen.
FIGSIZE_FULL = (6.8, 2.75)


In [2]:
RUN_DIR_NAME = "randomsearch_new_budgetloss_expertanalysis_reconstructed"
OUT_DIR_NAME = "figures_budgetloss_expertanalysis_reconstructed"


def finde_moe_root() -> Path:
    # Findet den moe-Ordner, egal ob das Notebook aus dem Repo-Root oder aus moe/plots gestartet wird.
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        for kandidat in (base, base / "moe", base / "single_pulse_classifier_training" / "moe"):
            if (kandidat / "moe_runs" / RUN_DIR_NAME).exists():
                return kandidat
    raise FileNotFoundError(f"moe_runs/{RUN_DIR_NAME} konnte vom aktuellen Arbeitsverzeichnis aus nicht gefunden werden.")


MOE_ROOT = finde_moe_root()
RUN_ROOT = MOE_ROOT / "moe_runs" / RUN_DIR_NAME
OUT_DIR = MOE_ROOT / "plots" / OUT_DIR_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

MOE_ROOT, RUN_ROOT, OUT_DIR

(PosixPath('/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/moe'),
 PosixPath('/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/moe/moe_runs/randomsearch_new_budgetloss_expertanalysis_reconstructed'),
 PosixPath('/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/moe/plots/figures_budgetloss_expertanalysis_reconstructed'))

In [ ]:
def dekodiere_float_token(token: str) -> str:
    return token.replace("p", ".").replace("em", "e-")


def parse_run_name(run_name: str) -> dict:
    muster = {
        "budget": r"budget(\d+)",
        "aux_ep": r"auxep(\d+)",
        "expert_lr": r"lr([^_]+)",
        "short": r"(_short)$",
    }
    meta = {"run": run_name}
    for key, pattern in muster.items():
        match = re.search(pattern, run_name)
        if not match:
            continue
        wert = match.group(1)
        if key in {"budget", "aux_ep"}:
            meta[key] = int(wert)
        elif key == "short":
            meta[key] = True
        else:
            meta[key] = dekodiere_float_token(wert)

    aux_ep = meta.get("aux_ep", "?")
    lr = meta.get("expert_lr", "?")
    meta["base_label"] = f"aux_ep={aux_ep}, lr={lr}"
    meta["label"] = meta["base_label"]
    return meta


def load_tensorboard_scalars(run_root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    zeilen = []
    run_zeilen = []

    for run_dir in sorted(p for p in run_root.iterdir() if p.is_dir()):
        tb_dir = run_dir / "tensorboard"
        meta = parse_run_name(run_dir.name)
        meta.update({"path": str(run_dir), "tensorboard_path": str(tb_dir)})

        if not tb_dir.exists():
            meta.update({"status": "TensorBoard-Ordner fehlt", "n_tags": 0, "n_points": 0})
            run_zeilen.append(meta)
            continue

        accumulator = EventAccumulator(str(tb_dir), size_guidance={"scalars": 0})
        try:
            accumulator.Reload()
        except Exception as exc:
            meta.update({"status": f"Lesefehler: {exc}", "n_tags": 0, "n_points": 0})
            run_zeilen.append(meta)
            continue

        tags = accumulator.Tags().get("scalars", [])
        n_points = 0
        for tag in tags:
            events = accumulator.Scalars(tag)
            n_points += len(events)
            for event in events:
                zeilen.append({
                    **meta,
                    "tag": tag,
                    "epoche": event.step,
                    "wall_time": event.wall_time,
                    "wert": float(event.value),
                })

        meta.update({"status": "ok" if tags else "keine Scalar-Tags", "n_tags": len(tags), "n_points": n_points})
        run_zeilen.append(meta)

    scalars = pd.DataFrame(zeilen)
    runs = pd.DataFrame(run_zeilen).sort_values(["status", "budget", "aux_ep", "expert_lr", "run"], na_position="last")
    return scalars, runs


scalars, runs = load_tensorboard_scalars(RUN_ROOT)
print(f"Geladen: {len(scalars):,} Scalar-Punkte aus {runs.query('n_points > 0').shape[0]} Durchläufen.")
display(runs[["label", "status", "n_tags", "n_points", "run"]])

In [4]:
ERWARTETE_TAGS = [
    "train/accuracy",
    "val_hard/accuracy",
    "val_hard/usage_small",
    "val_hard/usage_mid",
    "val_hard/usage_large",
    "val_topk/accuracy",
]

verfuegbare_tags = sorted(scalars["tag"].unique()) if not scalars.empty else []
fehlende_tags = [tag for tag in ERWARTETE_TAGS if tag not in verfuegbare_tags]
if fehlende_tags:
    warnings.warn("Folgende erwartete TensorBoard-Tags fehlen: " + ", ".join(fehlende_tags))

print("Verfuegbare Scalar-Tags:")
for tag in verfuegbare_tags:
    print(" -", tag)

Verfuegbare Scalar-Tags:
 - learning_rate/group_0
 - learning_rate/group_1
 - train/accuracy
 - train/budget
 - train/budget_loss_weight
 - train/budget_weight
 - train/ensemble
 - train/expert_aux
 - train/expert_aux_loss_weight
 - train/expert_aux_weight
 - train/expert_large
 - train/expert_mid
 - train/expert_small
 - train/only_aux_warmup
 - train/routed
 - train/soft_usage_large
 - train/soft_usage_mid
 - train/soft_usage_small
 - train/topk_noise_std
 - train/total
 - train/usage_large
 - train/usage_mid
 - train/usage_small
 - val_expert/aux
 - val_expert/aux_weight
 - val_expert/large
 - val_expert/large_accuracy
 - val_expert/large_loss
 - val_expert/mid
 - val_expert/mid_accuracy
 - val_expert/mid_loss
 - val_expert/small
 - val_expert/small_accuracy
 - val_expert/small_loss
 - val_hard/accuracy
 - val_hard/usage_large
 - val_hard/usage_mid
 - val_hard/usage_small
 - val_soft/accuracy
 - val_soft/budget
 - val_soft/budget_weight
 - val_soft/ensemble
 - val_soft/only_aux_warm

In [ ]:
gueltige_runs = runs.loc[runs["n_points"] > 0].copy()
gueltige_runs = gueltige_runs.sort_values(
    ["budget", "aux_ep", "expert_lr", "run"],
    na_position="last",
)
run_order = gueltige_runs["run"].tolist()


def format_number_de(value, precision=None):
    """Kompakte deutsche Zahlendarstellung für die Plot-Legenden."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "?"

    value = float(value)

    if precision is None:
        text = f"{value:g}"
    else:
        text = f"{value:.{precision}g}"

    # Wissenschaftliche Schreibweise kompakter machen.
    text = text.replace("e-0", "e-").replace("e+0", "e+")
    return text.replace(".", ",")


# Budget-Loss-Weight direkt aus TensorBoard lesen.
budget_weight_labels = {}
weight_frame = scalars.loc[
    scalars["tag"] == "train/budget_loss_weight",
    ["run", "wert"],
]

for run, values in weight_frame.groupby("run")["wert"]:
    unique_values = sorted({float(value) for value in values})

    if len(unique_values) == 1:
        budget_weight_labels[run] = format_number_de(unique_values[0])
    else:
        budget_weight_labels[run] = "→".join(
            format_number_de(value)
            for value in unique_values
        )


# Kompakte Legendenbezeichnung:
# - kein redundantes "Budget X"
# - kein "kurz"
# - LR auf drei signifikante Stellen gekürzt
def make_run_label(row):
    w_budget = budget_weight_labels.get(row["run"], "?")
    aux_ep = row.get("aux_ep", "?")

    try:
        lr = format_number_de(float(row["expert_lr"]), precision=3)
    except (TypeError, ValueError):
        lr = str(row.get("expert_lr", "?")).replace(".", ",")

    return (
        f"w_budget={w_budget}, "
        f"aux_ep={aux_ep}, "
        f"lr={lr}"
    )


gueltige_runs["label"] = gueltige_runs.apply(
    make_run_label,
    axis=1,
)

label_by_run = dict(
    zip(
        gueltige_runs["run"],
        gueltige_runs["label"],
    )
)

farben = plt.colormaps["tab20"].resampled(
    max(len(run_order), 1)
)

color_by_run = {
    run: farben(i)
    for i, run in enumerate(run_order)
}


USAGE_TAGS = {
    "val_hard/usage_small": r"$f_{\mathrm{small}}$",
    "val_hard/usage_mid": r"$f_{\mathrm{mid}}$",
    "val_hard/usage_large": r"$f_{\mathrm{large}}$",
}


TOPK_TAG_LABELS = {
    "val_topk/accuracy": "Genauigkeit",
    "val_topk/total": "Gesamt-Loss",
    "val_topk/ensemble": "Ensemble-Loss",
    "val_topk/expert_small": r"Loss $f_{\mathrm{small}}$",
    "val_topk/expert_mid": r"Loss $f_{\mathrm{mid}}$",
    "val_topk/expert_large": r"Loss $f_{\mathrm{large}}$",
    "val_topk/usage_small": r"Nutzung $f_{\mathrm{small}}$",
    "val_topk/usage_mid": r"Nutzung $f_{\mathrm{mid}}$",
    "val_topk/usage_large": r"Nutzung $f_{\mathrm{large}}$",
}


def tag_frame(tag: str) -> pd.DataFrame:
    frame = scalars.loc[
        scalars["tag"] == tag,
        ["run", "label", "epoche", "wert"],
    ].copy()

    return frame.sort_values(
        ["run", "epoche"]
    )


def ist_anteil(frame: pd.DataFrame) -> bool:
    if frame.empty:
        return False

    return frame["wert"].dropna().between(
        -0.02,
        1.02,
    ).all()


def style_axis(
    ax,
    ylabel=None,
    ylim=None,
    ytick_step=None,
    percent=False,
):
    ax.set_xlabel(
        "Epoche",
        labelpad=2,
    )

    if ylabel is not None:
        ax.set_ylabel(
            ylabel,
            labelpad=2,
        )

    if ylim is not None:
        ax.set_ylim(
            *ylim
        )

    if ytick_step is not None:
        ax.yaxis.set_major_locator(
            MultipleLocator(
                ytick_step
            )
        )

    if percent:
        ax.yaxis.set_major_formatter(
            PercentFormatter(
                xmax=1.0,
                decimals=0,
            )
        )

    ax.xaxis.set_major_locator(
        MaxNLocator(
            nbins=6,
            integer=True,
        )
    )

    ax.grid(
        axis="both",
        color="0.90",
        linewidth=0.7,
        linestyle="-",
    )

    ax.spines["top"].set_visible(
        False
    )

    ax.spines["right"].set_visible(
        False
    )


def speichere_abbildung(
    fig,
    dateiname: str,
):
    path = OUT_DIR / f"{dateiname}.pdf"

    fig.savefig(
        path
    )

    print(
        f"Gespeichert: "
        f"{path.relative_to(MOE_ROOT)}"
    )


def run_legend_handles(order):
    return [
        Line2D(
            [0],
            [0],
            color=color_by_run[run],
            linewidth=1.2,
            label=label_by_run[run],
        )
        for run in order
    ]


def add_run_legend(
    fig,
    order,
    fontsize=5.5,
):
    """
    Run-Legende unterhalb der gesamten Abbildung.
    Bei sieben Runs entstehen mit vier Spalten zwei Zeilen.
    """
    fig.legend(
        handles=run_legend_handles(order),
        loc="lower center",
        bbox_to_anchor=(0.5, 0.015),
        ncol=4,
        frameon=False,
        fontsize=fontsize,
        handlelength=1.0,
        handletextpad=0.25,
        columnspacing=0.55,
        borderaxespad=0.0,
        labelspacing=0.25,
    )


def beste_runs_nach_tag(tag: str) -> list[str]:
    frame = tag_frame(tag)
    if frame.empty:
        return run_order
    ranking = (
        frame.groupby("run", as_index=False)["wert"]
        .max()
        .sort_values("wert", ascending=False)
    )
    return ranking["run"].tolist()

## Genauigkeitsverläufe für Hard- und Top-k-Routing

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=FIGSIZE_FULL,
)

ax_hard, ax_topk = axes
run_order_accuracy = beste_runs_nach_tag("val_hard/accuracy")

for run in run_order_accuracy:
    train_frame = tag_frame("train/accuracy").loc[lambda df: df["run"] == run]
    val_frame = tag_frame("val_hard/accuracy").loc[lambda df: df["run"] == run]
    color = color_by_run[run]

    if not train_frame.empty:
        ax_hard.plot(
            train_frame["epoche"],
            train_frame["wert"],
            color=color,
            linestyle="--",
            linewidth=0.8,
            alpha=0.65,
            zorder=1,
        )

    if not val_frame.empty:
        ax_hard.plot(
            val_frame["epoche"],
            val_frame["wert"],
            color=color,
            linestyle="-",
            linewidth=0.9,
            alpha=0.85,
            zorder=2,
        )

ax_hard.set_title(
    "Hard-Routing",
    pad=3,
)

style_axis(
    ax_hard,
    ylabel="Genauigkeit",
    ylim=(0.50, 1.00),
    ytick_step=0.10,
    percent=True,
)

# Linienstile separat und kompakt im Plot erklären.
style_handles = [
    Line2D(
        [0], [0],
        color="0.25",
        linestyle="--",
        linewidth=0.9,
        label="Training",
    ),
    Line2D(
        [0], [0],
        color="0.25",
        linestyle="-",
        linewidth=0.9,
        label="Validierung",
    ),
]

ax_hard.legend(
    handles=style_handles,
    loc="lower right",
    frameon=True,
    framealpha=0.9,
    borderpad=0.3,
    labelspacing=0.25,
    handlelength=1.8,
    fontsize=6.3,
)

topk_frame = tag_frame("val_topk/accuracy")

for run in run_order_accuracy:
    run_frame = topk_frame.loc[topk_frame["run"] == run]

    if run_frame.empty:
        continue

    ax_topk.plot(
        run_frame["epoche"],
        run_frame["wert"],
        color=color_by_run[run],
        linewidth=0.9,
        alpha=0.85,
    )

ax_topk.set_title(
    "Top-k-Routing",
    pad=3,
)

style_axis(
    ax_topk,
    ylabel="Validierungsgenauigkeit",
    ylim=(0.60, 0.90),
    ytick_step=0.05,
    percent=True,
)

# Gemeinsame Run-Legende in zwei Zeilen unterhalb der Abbildung.
add_run_legend(
    fig,
    run_order_accuracy,
    fontsize=6.3,
)

fig.subplots_adjust(
    left=0.075,
    right=0.99,
    top=0.91,
    bottom=0.27,
    wspace=0.28,
)

speichere_abbildung(
    fig,
    "budgetloss_expertanalysis_reconstructed_hard_topk_accuracy",
)
plt.show()


## Nutzung der Klassifikatoren bei Hard-Routing

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=FIGSIZE_FULL,
    sharex=True,
    sharey=True,
)

run_order_usage = beste_runs_nach_tag("val_hard/accuracy")

for ax, (tag, title) in zip(axes, USAGE_TAGS.items()):
    frame = tag_frame(tag)

    for run in run_order_usage:
        run_frame = frame.loc[frame["run"] == run]

        if run_frame.empty:
            continue

        ax.plot(
            run_frame["epoche"],
            run_frame["wert"],
            color=color_by_run[run],
            linewidth=0.9,
            alpha=0.85,
        )

    ax.set_title(
        title,
        pad=3,
    )

    style_axis(
        ax,
        ylabel="Anteil" if ax is axes[0] else None,
        ylim=(0.0, 1.0),
        ytick_step=0.20,
        percent=True,
    )

add_run_legend(
    fig,
    run_order_usage,
    fontsize=6.3,
)

fig.subplots_adjust(
    left=0.075,
    right=0.99,
    top=0.91,
    bottom=0.27,
    wspace=0.22,
)

speichere_abbildung(
    fig,
    "budgetloss_expertanalysis_reconstructed_val_hard_classifier_usage",
)
plt.show()
